# Eyewear brand localization / OCR

This notebook runs the standalone `eyewear-localization/` pipeline. It first tests OCR independently, then optionally enables class-agnostic SAM3 eyewear localization. Brand names are never sent to SAM3.

## 1. Clone and install

The notebook installs both OCR and native SAM3 dependencies in a UV environment. It then downloads the gated SAM3 checkpoint directly over HTTP using an approved Hugging Face token from a Kaggle Secret named `HF_TOKEN` (no `hf auth login` required).

In [ ]:
import json, os, shutil, subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/fez-Ox/pxModel-Object-Counting.git"
REPO_DIR = Path("/kaggle/working/pxModel-localization")
APP_DIR = REPO_DIR / "eyewear-localization"
SAM3_APP = REPO_DIR / "sam3-verbose-counting"
SAM3_CHECKPOINT = SAM3_APP / "checkpoints" / "sam3.pt"

if not (REPO_DIR / ".git").is_dir():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
if shutil.which("uv") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)

# Install EasyOCR plus every dependency needed by the native SAM3 adapter.
subprocess.run(["uv", "sync", "--extra", "ocr", "--extra", "sam3"], cwd=APP_DIR, check=True)

def get_hf_token():
    token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_HUB_TOKEN")
    if token:
        return token
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        return None

# The downloader is direct HTTP and never writes the token to disk or runs `hf login`.
download_env = os.environ.copy()
hf_token = get_hf_token()
if hf_token:
    download_env["HF_TOKEN"] = hf_token
if not SAM3_CHECKPOINT.exists():
    download_command = [
        sys.executable, str(SAM3_APP / "download_model.py"),
        "--output", str(SAM3_CHECKPOINT),
        "--timeout", "600",
    ]
    subprocess.run(download_command, cwd=SAM3_APP, env=download_env, check=True)
else:
    print("Using existing checkpoint:", SAM3_CHECKPOINT)
print("Pipeline:", APP_DIR)
print("SAM3 checkpoint:", SAM3_CHECKPOINT)


## 2. Select an image

Edit `IMAGE_PATH` if the automatic first-image selection is not the image you want.

In [ ]:
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}
candidates = sorted(
    path for path in Path("/kaggle/input").rglob("*")
    if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
)
IMAGE_PATH = candidates[0] if candidates else Path("/kaggle/input/your-dataset/image.jpg")  # edit me
BRAND_FILE = APP_DIR / "brands.txt"  # edit or replace with your catalog
OUTPUT_DIR = APP_DIR / "outputs" / "notebook"
try:
    probe = subprocess.run(
        ["uv", "run", "python", "-c",
         "import torch; print('cuda' if torch.cuda.is_available() else 'cpu')"],
        cwd=APP_DIR, capture_output=True, text=True, check=True,
    )
    DEVICE = probe.stdout.strip()
except Exception:
    DEVICE = "cpu"

print("Image:", IMAGE_PATH)
print("Device:", DEVICE)
assert IMAGE_PATH.exists(), f"Update IMAGE_PATH: {IMAGE_PATH}"


## 3. OCR-only test

This stage does not need SAM3. It detects all text, matches the configured brand gazetteer, and writes `text_detections[]` and `signs[]`. A sign is not assigned to an eyewear instance at this stage.

In [ ]:
def run_pipeline(checkpoint=None):
    command = [
        "uv", "run", "python", "infer.py", str(IMAGE_PATH),
        "--brand-file", str(BRAND_FILE),
        "--ocr-backend", "easyocr",
        "--device", DEVICE,
        "--no-vlm-audit",
        "--out", str(OUTPUT_DIR),
    ]
    if checkpoint is not None:
        command += ["--sam3-checkpoint", str(checkpoint)]
    completed = subprocess.run(command, cwd=APP_DIR, text=True, capture_output=True)
    print(completed.stdout)
    if completed.returncode:
        print(completed.stderr)
        completed.check_returncode()
    result_path = OUTPUT_DIR / f"{IMAGE_PATH.stem}.json"
    return json.loads(result_path.read_text())

ocr_result = run_pipeline()
print("OCR text detections:")
for detection in ocr_result["text_detections"]:
    print(f"  {detection['text']!r} ({detection['confidence']:.2f})")
print("Gazetteer-matched signs:")
for sign in ocr_result["signs"]:
    print(f"  {sign['text']!r} -> {sign['brand']}")


In [ ]:
from IPython.display import display
from PIL import Image

display(Image.open(OUTPUT_DIR / f"{IMAGE_PATH.stem}.jpg"))


## 4. Full attribution test with SAM3

The setup cell has already downloaded the official checkpoint. It uses only class-agnostic eyewear prompts; brand names remain in the OCR/fusion stages.

In [ ]:
assert SAM3_CHECKPOINT.exists(), f"Checkpoint download failed: {SAM3_CHECKPOINT}"
full_result = run_pipeline(SAM3_CHECKPOINT)
print("Backends:", full_result["backends"])
print(f"Instances: {len(full_result['instances'])} | signs: {len(full_result['signs'])} | evidence: {len(full_result['evidence'])}")
print(f"Excluded by scene filter: {len(full_result['excluded_instances'])}")
for excluded in full_result["excluded_instances"]:
    print("  ", excluded["instance_id"], excluded["reasons"])
print("Per-instance decisions:")
for output in full_result["outputs"]:
    print(output["instance_id"], output["brand"], output["probabilities"])
display(Image.open(OUTPUT_DIR / f"{IMAGE_PATH.stem}.jpg"))


## 5. Inspect the JSON contract

The result keeps raw OCR, scoped signs, cue evidence, and final decisions separate for auditing.

In [ ]:
from pprint import pprint
pprint({
    "signs": ocr_result["signs"],
    "evidence": ocr_result["evidence"],
    "outputs": ocr_result["outputs"],
})
